# Collaborative Filtering Recommenders

Collaborative filtering uses only the user-course interactions, with no course content.
This notebook covers two tasks:

- **Part A, rating prediction:** predict the 3 to 5 rating a user gives a course, scored by test RMSE.
- **Part B, enrolment prediction:** rank the courses a user has not taken, scored by HitRate@10 and NDCG@10 on one held-out enrolment per user.

Both tasks use the shared, seeded splits in `recsys/splits.py`, so the results are directly comparable with the other notebooks.

In [1]:
import sys

import numpy as np
import pandas as pd
from surprise import NMF, SVD, BaselineOnly, Dataset, KNNBasic, Reader
from surprise.model_selection import GridSearchCV

sys.path.append("..")
from recsys import data, evaluation, metrics, recommenders, results, splits

## Part A: Rating Prediction

### Data and split
One random 80/20 split over interactions, shared by every rating model in the project.
The ratings have no timestamps, so a temporal split is not possible.

In [2]:
ratings = data.load_ratings()
train, test = splits.rating_split(ratings)

print(f"Ratings: {len(ratings):,} | users: {ratings['user'].nunique():,} | courses: {ratings['item'].nunique()}")
print(f"Train: {len(train):,} | test: {len(test):,}")
print(f"Test rows whose user has no training ratings: {(~test['user'].isin(train['user'])).mean():.1%}")
print(f"Global mean baseline test RMSE: {metrics.rmse(test['rating'], train['rating'].mean()):.4f}")

Ratings: 233,306 | users: 33,901 | courses: 126
Train: 186,644 | test: 46,662
Test rows whose user has no training ratings: 3.5%
Global mean baseline test RMSE: 0.8123


### Rating scale
The ratings are 3, 4 and 5, so the Surprise `Reader` is given `rating_scale=(3, 5)`.
The first version of this notebook used `rating_scale=(2, 3)`. Surprise clips every prediction to the rating scale,
so every prediction was capped at 3. That setting is reproduced at the end of Part A to show its effect.

In [3]:
reader = Reader(rating_scale=(data.RATING_MIN, data.RATING_MAX))
train_data = Dataset.load_from_df(train[["user", "item", "rating"]], reader)
trainset = train_data.build_full_trainset()
testset = list(test[["user", "item", "rating"]].itertuples(index=False, name=None))

### Models
Hyperparameters are chosen by 3-fold cross-validation on the training set only. The best configuration is then refit
on the full training set and scored once on the test set.

In [4]:
candidates = {
    "Baseline (user + item bias)": (BaselineOnly, {"verbose": [False]}),
    "KNN user-based": (
        KNNBasic,
        {"k": [10, 40], "sim_options": {"name": ["msd"], "user_based": [True]}, "verbose": [False]},
    ),
    "KNN item-based": (
        KNNBasic,
        {"k": [10, 40], "sim_options": {"name": ["msd", "cosine"], "user_based": [False]}, "verbose": [False]},
    ),
    "NMF": (
        NMF,
        {
            "n_factors": [15, 40],
            "n_epochs": [50],
            "reg_pu": [0.1],
            "reg_qi": [0.1],
            "init_low": [0.5],
            "init_high": [5.0],
            "random_state": [splits.SEED],
        },
    ),
    "SVD": (SVD, {"n_factors": [50, 100], "reg_all": [0.02, 0.1], "random_state": [splits.SEED]}),
}

rating_rows = []
for name, (algo_class, grid) in candidates.items():
    search = GridSearchCV(algo_class, grid, measures=["rmse"], cv=3, n_jobs=1)
    search.fit(train_data)
    best_params = search.best_params["rmse"]

    model = algo_class(**best_params)
    model.fit(trainset)
    predictions = model.test(testset)
    test_rmse = metrics.rmse([p.r_ui for p in predictions], [p.est for p in predictions])

    rating_rows.append({"family": "Collaborative filtering", "model": name, "test_rmse": test_rmse})
    shown = {k: v for k, v in best_params.items() if k not in ("verbose", "random_state")}
    print(f"{name:<28} CV RMSE {search.best_score['rmse']:.4f} | test RMSE {test_rmse:.4f} | {shown}")

Baseline (user + item bias)  CV RMSE 0.8220 | test RMSE 0.8181 | {}


KNN user-based               CV RMSE 0.8272 | test RMSE 0.8222 | {'k': 40, 'sim_options': {'name': 'msd', 'user_based': True}}


KNN item-based               CV RMSE 0.9217 | test RMSE 0.8786 | {'k': 40, 'sim_options': {'name': 'cosine', 'user_based': False}}


NMF                          CV RMSE 0.9589 | test RMSE 0.9391 | {'n_factors': 40, 'n_epochs': 50, 'reg_pu': 0.1, 'reg_qi': 0.1, 'init_low': 0.5, 'init_high': 5.0}


SVD                          CV RMSE 0.8289 | test RMSE 0.8253 | {'n_factors': 50, 'reg_all': 0.1}


In [5]:
pd.DataFrame(rating_rows).sort_values("test_rmse").round(4)

,family,model,test_rmse
0,Collaborative filtering,Baseline (user + item bias),0.8181
1,Collaborative filtering,KNN user-based,0.8222
4,Collaborative filtering,SVD,0.8253
2,Collaborative filtering,KNN item-based,0.8786
3,Collaborative filtering,NMF,0.9391


### The original rating-scale setting
Same split, item-based KNN, but with `rating_scale=(2, 3)` as in the first version of this notebook.

In [6]:
wrong_reader = Reader(rating_scale=(2, 3))
wrong_trainset = Dataset.load_from_df(train[["user", "item", "rating"]], wrong_reader).build_full_trainset()

model = KNNBasic(k=10, sim_options={"name": "msd", "user_based": False}, verbose=False)
model.fit(wrong_trainset)
estimates = np.array([p.est for p in model.test(testset)])

print(f"Largest prediction: {estimates.max():.2f}")
print(f"Test RMSE: {metrics.rmse(test['rating'], estimates):.4f}")
print(f"RMSE of always predicting 3: {metrics.rmse(test['rating'], 3):.4f}")

Largest prediction: 3.00
Test RMSE: 1.2901
RMSE of always predicting 3: 1.2901


### Part A conclusion
With the correct rating scale no collaborative filtering model beats the global mean baseline.
`model_comparison.ipynb` shows why: the ratings do not depend on the user or the course, so there is nothing for a
rating model to learn. The original 1.29 RMSE came from the clipping bug, not from the models.

In [7]:
results.save(rating_rows, "rating_rmse", source="collaborative_filtering");

## Part B: Enrolment Prediction

Which courses a user enrols in does carry signal, even though the ratings do not.
For every user with at least two enrolments, one random enrolment is held out as the test item.
Each model ranks all courses the user has not taken in training, and we check whether the held-out course is in the top 10.

Hyperparameters are selected on a second held-out enrolment taken from the training data (validation), never on the test set.

In [8]:
index = data.Index(ratings)
train_loo, test_loo = splits.leave_one_out(ratings)
fit_loo, val_loo = splits.leave_one_out(train_loo, seed=splits.SEED + 1)

print(f"Test users: {len(test_loo):,} | validation users: {len(val_loo):,} | candidate courses: {index.n_items}")

Test users: 25,581 | validation users: 25,199 | candidate courses: 126


### Neighbourhood and matrix factorisation models
Item KNN and user KNN use cosine similarity on the binary user-course enrolment matrix.
PureSVD is a truncated SVD of the same matrix.

In [9]:
def item_knn(k):
    return lambda matrix, users: recommenders.item_knn_scores(
        matrix, users, recommenders.item_cosine_similarity(matrix), k
    )


def user_knn(k):
    return lambda matrix, users: recommenders.user_knn_scores(matrix, users, k)


def pure_svd(rank):
    return lambda matrix, users: recommenders.pure_svd_scores(matrix, users, rank)


best_item_k = evaluation.select_on_validation(item_knn, [10, 20, 50, 125], fit_loo, val_loo, index, "Item KNN k")
best_user_k = evaluation.select_on_validation(user_knn, [50, 100, 200, 400, 800], fit_loo, val_loo, index, "User KNN k")
best_rank = evaluation.select_on_validation(pure_svd, [2, 4, 8, 16], fit_loo, val_loo, index, "PureSVD rank")

Item KNN k validation NDCG@10 -> 10: 0.4506, 20: 0.4585, 50: 0.4649, 125: 0.4665 | selected 125


User KNN k validation NDCG@10 -> 50: 0.4557, 100: 0.4847, 200: 0.4905, 400: 0.4888, 800: 0.4830 | selected 200


PureSVD rank validation NDCG@10 -> 2: 0.3774, 4: 0.4171, 8: 0.3529, 16: 0.2723 | selected 4


### Ranking by predicted rating
As a control, a Surprise SVD rating model is trained on the training ratings and courses are ranked by predicted rating.
If the ratings carry no signal, this should do no better than chance. It is saved as a baseline, not as a collaborative
filtering model, so that it does not distort the family averages.

In [10]:
def rating_svd_scores(fit_df, eval_users):
    reader = Reader(rating_scale=(data.RATING_MIN, data.RATING_MAX))
    trainset = Dataset.load_from_df(fit_df[["user", "item", "rating"]], reader).build_full_trainset()
    model = SVD(random_state=splits.SEED)
    model.fit(trainset)

    raw_to_inner = {trainset.to_raw_iid(inner): inner for inner in trainset.all_items()}
    item_inner = np.array([raw_to_inner.get(item, -1) for item in index.items])
    known = item_inner >= 0
    item_bias = np.where(known, model.bi[item_inner], 0.0)
    item_factors = np.where(known[:, None], model.qi[item_inner], 0.0)

    scores = np.zeros((len(eval_users), index.n_items))
    for row, user in enumerate(index.users[eval_users]):
        inner = trainset.to_inner_uid(user)
        scores[row] = model.bu[inner] + item_bias + item_factors @ model.pu[inner]
    return scores + trainset.global_mean


def evaluate_rating_svd(fit_df, eval_df):
    users, targets = index.encode(eval_df)
    matrix = splits.interaction_matrix(fit_df, index)
    scores = rating_svd_scores(fit_df, users)
    return metrics.ranking_metrics(scores, matrix[users].toarray() > 0, targets)

### Test results

In [11]:
ranking_models = {
    f"Item KNN (k={best_item_k})": item_knn(best_item_k),
    f"User KNN (k={best_user_k})": user_knn(best_user_k),
    f"PureSVD (rank={best_rank})": pure_svd(best_rank),
}

ranking_rows = []
for name, score_fn in ranking_models.items():
    scores = evaluation.evaluate_ranking(score_fn, train_loo, test_loo, index)
    ranking_rows.append({"family": "Collaborative filtering", "model": name, **scores})

scores = evaluate_rating_svd(train_loo, test_loo)
ranking_rows.append({"family": "Baseline", "model": "SVD on ratings (ranked by predicted rating)", **scores})

pd.DataFrame(ranking_rows).sort_values("ndcg@10", ascending=False).round(4)

,family,model,hit_rate@10,ndcg@10
1,Collaborative filtering,User KNN (k=200),0.7856,0.5630
0,Collaborative filtering,Item KNN (k=125),0.7379,0.5248
2,Collaborative filtering,PureSVD (rank=4),0.6946,0.4712
3,Baseline,SVD on ratings (ranked by predicted rating),0.0857,0.0398


In [12]:
results.save(ranking_rows, "ranking", source="collaborative_filtering");

### Part B conclusion
Neighbourhood methods on the enrolment matrix recommend well. They are compared with the popularity baseline and the
other families in `model_comparison.ipynb`. Ranking by predicted rating does no better than random, which agrees with Part A:
the useful signal is in which courses people take, not in how they rate them.